# 📊 Análise de Qualidade de Dados

Este notebook demonstra o uso da Plataforma de Qualidade de Dados para:
- Validar schemas com Pandera
- Aplicar regras de negócio customizadas
- Gerar profiling detalhado
- Criar relatórios de qualidade
- Configurar alertas automáticos

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

from src.validators.schema_validator import SchemaValidator, ResultadoValidacao
from src.validators.data_profiler import DataProfiler
from src.reporters.quality_report import QualityReport
from src.reporters.alerts import AlertSystem, NivelAlerta

## 1. Criando Dados de Exemplo

Vamos criar datasets de exemplo com ~5% de sujeira proposital (seed fixa) para a demo falhar de verdade.

In [ ]:
random.seed(42)
np.random.seed(42)

def gerar_dados_clientes(n: int = 100) -> pd.DataFrame:
    """Gera dados fictícios de clientes com ~5% de sujeira proposital."""
    df = pd.DataFrame({
        'id_cliente': range(1, n + 1),
        'nome': [f'Cliente {i}' for i in range(1, n + 1)],
        'email': [f'cliente{i}@email.com' for i in range(1, n + 1)],
        'telefone': [f'(11) 9{random.randint(1000, 9999)}-{random.randint(1000, 9999)}' for _ in range(n)],
        'data_cadastro': [datetime.now() - timedelta(days=random.randint(1, 365)) for _ in range(n)],
        'ativo': [random.choice([True, False]) for _ in range(n)]
    })
    # sujeira: 2 ids duplicados, 2 emails inválidos, 1 nome curto
    df.loc[5, 'id_cliente'] = df.loc[0, 'id_cliente']
    df.loc[42, 'id_cliente'] = df.loc[10, 'id_cliente']
    df.loc[7, 'email'] = 'invalido'
    df.loc[63, 'email'] = 'sem-arroba'
    df.loc[88, 'nome'] = 'X'
    return df

def gerar_dados_pedidos(n: int = 200) -> pd.DataFrame:
    """Gera dados fictícios de pedidos com ~5% de sujeira proposital."""
    statuses = ['pendente', 'processando', 'enviado', 'entregue', 'cancelado']
    df = pd.DataFrame({
        'id_pedido': range(1, n + 1),
        'id_cliente': [random.randint(1, 100) for _ in range(n)],
        'data_pedido': [datetime.now() - timedelta(days=random.randint(1, 90)) for _ in range(n)],
        'valor_total': [round(random.uniform(10, 5000), 2) for _ in range(n)],
        'status': [random.choice(statuses) for _ in range(n)]
    })
    # sujeira: 5 valores negativos, 5 status fora do domínio
    df.loc[10:14, 'valor_total'] = -50.0
    df.loc[100:104, 'status'] = 'extraviado'
    return df

df_clientes = gerar_dados_clientes(100)
df_pedidos = gerar_dados_pedidos(200)

print(f"Clientes gerados: {len(df_clientes)} registros")
print(f"Pedidos gerados: {len(df_pedidos)} registros")


In [ ]:
# Visualizar primeiras linhas
print("=== Clientes ===")
display(df_clientes.head())

print("\n=== Pedidos ===")
display(df_pedidos.head())

## 2. Validação de Schemas com Pandera

Vamos validar a estrutura dos dados usando schemas Pandera.

In [ ]:
# Inicializar validador
validator = SchemaValidator()

# Validar DataFrame de clientes
resultado_clientes = validator.validar_dataframe(df_clientes, 'cliente')
print(f"Validação de clientes: {resultado_clientes}")

if not resultado_clientes.valido:
    print("Erros encontrados:")
    for erro in resultado_clientes.erros:
        print(f"  - {erro}")

In [ ]:
# Validar DataFrame de pedidos
resultado_pedidos = validator.validar_dataframe(df_pedidos, 'pedido')
print(f"Validação de pedidos: {resultado_pedidos}")

In [ ]:
# Criar schema customizado
schema_customizado = validator.criar_schema_customizado(
    colunas={
        'id': {'dtype': 'int64', 'nullable': False, 'unique': True},
        'nome': {'dtype': 'object', 'nullable': False},
        'valor': {'dtype': 'float64', 'nullable': False, 'ge': 0}
    }
)

df_teste = pd.DataFrame({
    'id': [1, 2, 3],
    'nome': ['A', 'B', 'C'],
    'valor': [10.5, 20.0, 30.5]
})

resultado = schema_customizado.validate(df_teste, lazy=True)
print(f"Schema customizado válido: {resultado is not None}")

## 3. Profiling de Dados

Vamos gerar um perfil detalhado dos dados.

In [ ]:
# Inicializar profiler
profiler = DataProfiler()

# Gerar perfil dos clientes
perfil_clientes = profiler.perfilar(df_clientes, nome='clientes')

print(f"\n=== Perfil de Clientes ===")
print(f"Total de registros: {perfil_clientes.total_registros}")
print(f"Total de colunas: {perfil_clientes.total_colunas}")
print(f"Score de qualidade: {perfil_clientes.score_qualidade}%")
print(f"Duplicatas: {perfil_clientes.duplicatas_totais}")
print(f"Colunas com nulos: {perfil_clientes.colunas_com_nulos}")

In [ ]:
# Detalhes por coluna
for nome, col in perfil_clientes.colunas.items():
    print(f"\n--- {nome} ---")
    print(f"  Tipo: {col.tipo}")
    print(f"  Nulos: {col.nulos} ({col.percentual_nulos}%)")
    print(f"  Únicos: {col.unicos}")
    if col.minimo is not None:
        print(f"  Mínimo: {col.minimo}")
        print(f"  Máximo: {col.maximo}")
    if col.media is not None:
        print(f"  Média: {col.media:.2f}")

In [ ]:
# Gerar resumo executivo
resumo = profiler.gerar_resumo(perfil_clientes)
print("\n=== Resumo Executivo ===")
for chave, valor in resumo.items():
    if chave != 'problemas_encontrados':
        print(f"{chave}: {valor}")

if resumo['problemas_encontrados']:
    print("\nProblemas encontrados:")
    for problema in resumo['problemas_encontrados']:
        print(f"  ⚠️ {problema}")

## 4. Detecção de Anomalias

Vamos detectar outliers nos valores dos pedidos.

In [ ]:
# Detectar anomalias no valor_total usando IQR
anomalias = profiler.detectar_anomalias(df_pedidos, 'valor_total', metodo='iqr', limiar=1.5)

print(f"Anomalias detectadas: {len(anomalias)} de {len(df_pedidos)} registros")
if len(anomalias) > 0:
    print("\nRegistros com valores atípicos:")
    display(anomalias[['id_pedido', 'valor_total']].head(10))

In [ ]:
# Detectar anomalias usando Z-Score
anomalias_zscore = profiler.detectar_anomalias(df_pedidos, 'valor_total', metodo='zscore', limiar=2)

print(f"Anomalias (Z-Score): {len(anomalias_zscore)} registros")

## 5. Geração de Relatórios

Vamos gerar relatórios HTML e JSON com os resultados.

In [ ]:
# Inicializar gerador de relatórios
report = QualityReport()

# Preparar resultados para o relatório
resultados = {
    'dataset': 'clientes',
    'score_qualidade': perfil_clientes.score_qualidade,
    'total_registros': perfil_clientes.total_registros,
    'colunas_analisadas': perfil_clientes.total_colunas,
    'problemas': resumo.get('problemas_encontrados', []),
    'validacoes': [
        {'nome': 'Schema Pandera', 'sucesso': resultado_clientes.valido, 'detalhes': 'Schema válido' if resultado_clientes.valido else 'Schema inválido'},
    ]
}

# Gerar relatório HTML
caminho_html = report.gerar_html(resultados, 'relatorio_qualidade.html')
print(f"Relatório HTML gerado: {caminho_html}")

# Gerar relatório JSON
caminho_json = report.gerar_json(resultados, 'relatorio_qualidade.json')
print(f"Relatório JSON gerado: {caminho_json}")

In [ ]:
# Gerar resumo executivo
perfil_pedidos = profiler.perfilar(df_pedidos, nome='pedidos')

resumo_executivo = report.gerar_resumo_executivo([
    profiler.gerar_resumo(perfil_clientes),
    profiler.gerar_resumo(perfil_pedidos)
])

print("\n=== Resumo Executivo ===")
print(f"Total de datasets: {resumo_executivo['total_datasets']}")
print(f"Total de registros: {resumo_executivo['total_registros']:,}")
print(f"Score médio: {resumo_executivo['score_qualidade_medio']}%")

# Gerar HTML do resumo
report.gerar_html_resumo_executivo(resumo_executivo, 'resumo_executivo.html')
print("\nResumo executivo HTML gerado: resumo_executivo.html")

## 6. Sistema de Alertas

Vamos configurar e testar o sistema de alertas.

In [ ]:
# Inicializar sistema de alertas
alertas = AlertSystem()

# Verificar falhas nos resultados
alertas_gerados = alertas.verificar_falhas(resultados)

print(f"Alertas gerados: {len(alertas_gerados)}")
for alerta in alertas_gerados:
    print(f"\n[{alerta.nivel.value.upper()}] {alerta.titulo}")
    print(f"  Mensagem: {alerta.mensagem}")
    print(f"  Tipo: {alerta.tipo.value}")

In [ ]:
# Estatísticas dos alertas
estatisticas = alertas.obter_estatisticas()
print("\n=== Estatísticas de Alertas ===")
for chave, valor in estatisticas.items():
    print(f"{chave}: {valor}")

In [ ]:
# Testar com dados problemáticos
df_problematico = df_clientes.copy()
df_problematico.loc[0:10, 'email'] = None  # Adicionar nulos
df_problematico.loc[20:25, 'nome'] = 'A'  # Nomes curtos

resultado_problematico = validator.validar_dataframe(df_problematico, 'cliente')
print(f"Validação com dados problemáticos: {resultado_problematico}")

if not resultado_problematico.valido:
    print("\nErros encontrados:")
    for erro in resultado_problematico.erros[:5]:  # Primeiros 5 erros
        print(f"  - {erro}")

In [ ]:
# Quarentena: mesmas regras do pipeline (DQ-03: validated/ = aprovado Pandera)
from src.pipeline import dividir_validados_rejeitados

validados, rejeitados = dividir_validados_rejeitados(df_clientes, resultado_clientes)
print(f"Validados: {len(validados)} | Rejeitados: {len(rejeitados)} de {len(df_clientes)}")
if len(rejeitados):
    display(rejeitados.head())


## 7. Comparação de Dados

Vamos comparar dois datasets para identificar mudanças.

In [ ]:
# Criar "versão anterior" dos dados
df_anterior = df_clientes.copy()
df_anterior = df_anterior.drop(columns=['telefone'])  # Remover coluna
df_anterior.loc[0:5, 'nome'] = None  # Adicionar nulos

# Perfilar ambas as versões
perfil_anterior = profiler.perfilar(df_anterior, nome='clientes_anterior')
perfil_atual = profiler.perfilar(df_clientes, nome='clientes_atual')

# Comparar perfis
comparacao = profiler.comparar_perfis(perfil_anterior, perfil_atual)

print("=== Comparação de Dados ===")
print(f"Diferença de registros: {comparacao['registros_diferenca']}")
print(f"Diferença de colunas: {comparacao['colunas_diferenca']}")
print(f"Mudança no score: {comparacao['score_diferenca']:.2f}")

if comparacao['colunas_adicionadas']:
    print(f"\nColunas adicionadas: {comparacao['colunas_adicionadas']}")
if comparacao['colunas_removidas']:
    print(f"Colunas removidas: {comparacao['colunas_removidas']}")

## 📋 Resumo

Neste notebook demonstramos:

1. **Validação de Schemas** - Usando Pandera para validar estrutura e tipos
2. **Profiling de Dados** - Análise detalhada de cada coluna
3. **Detecção de Anomalias** - Identificação de outliers com IQR e Z-Score
4. **Relatórios** - Geração de HTML e JSON
5. **Alertas** - Sistema automático de notificações
6. **Comparação** - Análise de mudanças entre versões

### Próximos Passos

- Calibrar o score de qualidade com a taxa de aprovação de linhas
- Configurar alertas via email ou webhook
- Automatizar execução com cron jobs
- Conectar com bancos de dados via SQLAlchemy